# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Deployed paper:** https://haideriqbal499.github.io/Flyrank_Ml_internship/paper/  
**Lane 2 — Refresh / Content Opportunity Scoring.**

This notebook mirrors the paper: question → data → method → results → limits → recommendations → artifacts.  
Design notes (research before build): one HTML landing page per paper (Distill/ARCH-style — semantic HTML, mobile reading column, figures with captions, content works without JS); abstract first; acknowledgments + data credit last.

> Skills: `writing-research-papers` + `deploying-static-pages`.

## 1. Question

**Research question:** Which content pages should an SEO/content editor review first for refresh, expansion, protection, pruning, or monitoring?

**FlyRank case (same story as the paper):** content that ranks can quietly decay across a large inventory; hand-written product flags help but still leave "which URL first?" — that decision is this lane's case study.

| Piece | Choice |
|---|---|
| Decision | What to open first when capacity is limited |
| Unit | One content item (page) |
| Output | Ranked queue + suggested action + reason codes |
| Wrong-call cost | False positive wastes edit hours; false negative misses a visible sliding URL |
| Why ML helps | Demand, position, CTR, age, and engagement interact — a single rule buries signal in noise |

This is **decision support**, not a claim that a refresh recovers traffic.


In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/haideriqbal499/Flyrank_Ml_internship"
REPO_DIR = "Flyrank_Ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.abspath(os.sep):
        os.chdir("..")

OUT = Path("work/outputs")
FIG = Path("work/figures")
PAPER_IMG = Path("docs/paper/img")
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

QUESTION = (
    "Which content pages should an editor review first for refresh, "
    "expansion, protection, pruning, or monitoring?"
)
print("Research question:", QUESTION)
print("Deployed paper URL:", "https://haideriqbal499.github.io/Flyrank_Ml_internship/paper/")
print("Working dir:", os.getcwd())

Research question: Which content pages should an editor review first for refresh, expansion, protection, pruning, or monitoring?
Deployed paper URL: https://haideriqbal499.github.io/Flyrank_Ml_internship/paper/
Working dir: C:\Users\Windows 11\Flyrank_Ml_internship


## 2. Data

**Starter (validated metrics):** 30,000 anonymized rows, 32 clients, trailing-90d metrics.  
**Warehouse (contract + scale path):** `FlyRank/internship-warehouse` — ~79M daily rows (2025-01-27 → 2026-06-30), plus dims. Used to write a public-safe contract; sealed final month not used for label development.

**Excluded from X:** `trend_*` / label (trap), IDs as features, titles/URLs/domains/raw queries, misaligned query-table windows, GA4 zeros without availability.

In [2]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

print(f"Starter rows: {len(df):,}")
print(f"Clients (pseudonym): {df['client_id'].nunique()}")
print(f"Declining-label rate: {df['is_declining_label'].mean():.3f}")
print("Warehouse (documented): ~79M daily fact rows, Jan 2025–Jun 2026")
print("Public-safe check: no client names / domains / raw queries printed.")

Starter rows: 30,000
Clients (pseudonym): 32
Declining-label rate: 0.542
Warehouse (documented): ~79M daily fact rows, Jan 2025–Jun 2026
Public-safe check: no client names / domains / raw queries printed.


## 3. Methodology

- **Label proxy:** `trend_direction == down` (snapshot — not future window).
- **Baseline:** CTR-gap rule (imp≥500, pos 1–20, ctr&lt;0.5).
- **Models:** Logistic regression + Random Forest; features exclude trend/label.
- **Split:** client holdout (~20% clients).
- **Leakage:** trend-in-X attack → perfect AUC (trap confirmed).

In [3]:
w05 = json.loads((OUT / "w05_model_vs_baseline.json").read_text(encoding="utf-8"))
w06 = json.loads((OUT / "w06_validation_audit.json").read_text(encoding="utf-8"))

print("Split:", w05["split"], "| random_state:", w05["random_state"])
print("Test rows:", w05["test_rows"], "| test clients:", w05["test_clients"], "| base rate:", w05["base_rate_test"])
print("Features:", ", ".join(w05["features"]))
print("\nLeakage attack (W06):")
print("  honest P@50:", w06["leakage_attack"]["honest"]["precision@50"])
print("  with trend_pct P@50:", w06["leakage_attack"]["with_trend_pct"]["precision@50"], "(trap)")

Split: client_holdout | random_state: 42
Test rows: 2325 | test clients: 6 | base rate: 0.391
Features: log_impressions_90d, log_clicks_90d, log_sessions_90d, days_with_impressions, days_with_sessions, content_age_days, days_since_last_update, ctr, avg_position, has_position, engagement_rate, scroll_rate, word_count_filled, has_word_count

Leakage attack (W06):
  honest P@50: 0.9
  with trend_pct P@50: 1.0 (trap)


## 4. Results (vs baseline)

Same client-holdout split. Random forest Precision@50 = **0.900** vs CTR-gap baseline **0.300** (test base rate 0.391). Directional ranking lift under the proxy — not causal recovery evidence.

In [4]:
cmp = pd.DataFrame(w05["comparison"])
print(cmp.to_string(index=False))
print("\nSafe claim:")
print(w06["safe_claim"])

# Rebuild the paper chart from receipts
subprocess.run([sys.executable, "work/scripts/build_paper_charts.py"], check=True)
print("\nChart written to work/figures/ and docs/paper/img/")

                method  precision@10  precision@50  roc_auc  avg_precision
week4_ctr_gap_baseline           0.1          0.30   0.5572         0.4097
   logistic_regression           0.6          0.54   0.7145         0.5501
         random_forest           0.9          0.90   0.7641         0.6684

Safe claim:
On a client-holdout of the starter snapshot (test n=2,325, 6 unseen clients), the random forest measured Precision@50=0.900 and ROC AUC=0.764 against the declining-trend proxy (test base rate=0.391; random-split Precision@50 was 0.960). That is a directional, decision-support ranking signal for which pages look worth reviewing first -- not causal evidence that a refresh recovers traffic, and not a claim about Google's algorithm.

Chart written to work/figures/ and docs/paper/img/


## 5. Limitations

Proxy label (not future window); no causal design; metrics on 30k/32 clients not full 79M; holdout membership sensitivity; staleness OPPOSITE on this snapshot; not Google’s algorithm.

In [5]:
limits = [
    "Snapshot trend proxy ≠ sealed next-window decline",
    "No experiment → no causal 'refresh recovers traffic' claim",
    "Holdout metrics are starter-scale (30k / 32 clients), not full warehouse",
    "Six test clients → Precision@50 can move with membership",
    "Deep-stale age alone was OPPOSITE — do not automate from age",
    "Not a model of Google's ranking system",
]
for i, line in enumerate(limits, 1):
    print(f"{i}. {line}")
print("\nClaim ladder words:", ", ".join(w06["claim_language"]))

1. Snapshot trend proxy ≠ sealed next-window decline
2. No experiment → no causal 'refresh recovers traffic' claim
3. Holdout metrics are starter-scale (30k / 32 clients), not full warehouse
4. Six test clients → Precision@50 can move with membership
5. Deep-stale age alone was OPPOSITE — do not automate from age
6. Not a model of Google's ranking system

Claim ladder words: observed, measured, directional, decision-support


## 6. Ranked recommendations

From the W07 action playbook: blended score, reason codes, archetype→action map, confidence bands, and an explicit no-go list (no auto-publish / auto-prune).

In [6]:
w07 = json.loads((OUT / "w07_action_playbook_metrics.json").read_text(encoding="utf-8"))
print("Rows scored:", w07["rows_scored"])
print("Holdout P@50:", w07["holdout_metrics"]["precision@50"])
print("\nAction counts:")
for k, v in w07["action_counts"].items():
    print(f"  {k}: {v:,}")
print("\nNo-go:")
for item in w07["no_go"]:
    print(" -", item)
print("\nPlaybook claim:")
print(w07["safe_claim"])

Rows scored: 30000
Holdout P@50: 0.9

Action counts:
  refresh: 11,717
  monitor: 9,725
  refresh_and_review_ctr: 6,296
  refresh_and_review_engagement: 1,879
  protect: 301
  expand_and_refresh: 82

No-go:
 - Do not auto-publish edits from this queue
 - Do not auto-prune or redirect from this queue
 - Do not claim refresh recovers traffic from this observational score
 - Do not use trend_pct/trend_direction as model features
 - Do not export client names, domains, raw queries, or private URLs
 - Do not trigger refreshes from age alone on this snapshot

Playbook claim:
On this starter snapshot, the playbook ranks pages for human refresh review using a blended RF+baseline score (client-holdout Precision@50=0.900). It is directional decision-support with reason codes — not causal evidence that a refresh recovers traffic, and not a Google-algorithm prediction.


## 7. Artifacts the paper embeds

Charts live under `docs/paper/img/` (GitHub Pages) and `work/figures/` (repo receipts). Paper URL is recorded in `submission/paper_url.txt`.

In [7]:
PAPER_URL = "https://haideriqbal499.github.io/Flyrank_Ml_internship/paper/"
url_path = Path("submission/paper_url.txt")
url_path.write_text(PAPER_URL + "\n", encoding="utf-8")

artifacts = [
    PAPER_IMG / "model_vs_baseline.svg",
    PAPER_IMG / "action_mix.svg",
    PAPER_IMG / "top_reason_codes.svg",
    PAPER_IMG / "top_feature_importance.svg",
    PAPER_IMG / "archetype_action.svg",
    Path("docs/paper/index.html"),
    url_path,
]
print("Paper URL written to", url_path, "→", url_path.read_text(encoding="utf-8").strip())
for p in artifacts:
    print(f"  {'OK' if p.exists() else 'MISSING'}: {p}")

capstone_metrics = {
    "paper_url": PAPER_URL,
    "question": QUESTION,
    "headline_result": {
        "split": "client_holdout",
        "rf_precision@50": 0.9,
        "baseline_precision@50": 0.3,
        "test_base_rate": 0.391,
        "rf_roc_auc": 0.7641,
    },
    "safe_claim": w06["safe_claim"],
    "sections": [
        "abstract",
        "introduction",
        "data",
        "methodology",
        "results",
        "limitations",
        "ranked_recommendations",
        "reproducibility",
        "acknowledgments_flyrank_ai",
    ],
}
(OUT / "capstone_paper_metrics.json").write_text(json.dumps(capstone_metrics, indent=2), encoding="utf-8")
print("Wrote work/outputs/capstone_paper_metrics.json")

Paper URL written to submission\paper_url.txt → https://haideriqbal499.github.io/Flyrank_Ml_internship/paper/
  OK: docs\paper\img\model_vs_baseline.svg
  OK: docs\paper\img\action_mix.svg
  OK: docs\paper\img\top_reason_codes.svg
  OK: docs\paper\img\top_feature_importance.svg
  OK: docs\paper\img\archetype_action.svg
  OK: docs\paper\index.html
  OK: submission\paper_url.txt
Wrote work/outputs/capstone_paper_metrics.json


## ML-12 — 5-minute demo outline + shareable cuts

Ready for an optional Week-8 showcase. Paper (case study lives inside it):  
https://haideriqbal499.github.io/Flyrank_Ml_internship/paper/

Also mirrored at `work/ml12_demo_and_cuts.md`.

### 5-minute demo outline

| Beat | Time | What you say / show |
|---|---|---|
| **1. Question** | ~45s | FlyRank content problem: pages decay in a large inventory; hand-written flags still leave "which URL first?" Open with that decision — not the model. |
| **2. Method** | ~75s | Transparent CTR-gap rule as baseline → logistic regression + random forest; **no** `trend_*` features; **client holdout** (~20% clients). One sentence on the leakage trap (add `trend_pct` → fake-perfect scores). |
| **3. One chart** | ~60s | Show **Figure 1** (`model_vs_baseline.svg` / Precision@50 bars). Point at the three bars only — same split. |
| **4. One honest result** | ~60s | Client-holdout: RF Precision@50 **0.90** vs CTR-gap rule **0.30** (test n=2,325, base rate 0.391). Say: ranking quality under a **snapshot decline proxy** — not proof a refresh recovers traffic. |
| **5. One recommendation** | ~60s | Playbook: open **high-confidence, high-demand** rows first; use reason codes; **never** auto-publish or auto-prune from the score. Close with paper URL + honest limits. |

**Props:** paper live URL · Figure 1 · top of playbook (action + reason code). Skip deep feature lists if time is tight.

---

### Shareable cut A — social post (methodology)

> How I ranked FlyRank-style refresh candidates without cheating the label: a transparent CTR-gap rule as baseline, then a random forest on demand/visibility features only — no `trend_*` in X — scored with **client holdout** Precision@K. Chart: Precision@50 0.90 (RF) vs 0.30 (rule) on the same split. Decision support with reason codes, not "we predicted Google."  
> Paper: https://haideriqbal499.github.io/Flyrank_Ml_internship/paper/  
> (Attach: Precision@50 bar chart)

---

### Shareable cut B — employer-facing summary (3 sentences)

I built a ranked review queue that tells content editors which pages to open first when a FlyRank-scale inventory is too large to scan by hand. I trained and validated on the FlyRank ML Internship dataset — 30,000 anonymized starter pages for holdout metrics, with the ~79M-row warehouse release used for the public-safe data contract. On a client holdout, a random forest measured Precision@50 of 0.90 versus 0.30 for a transparent CTR-gap rule: directional decision-support with reason codes, not causal proof that a refresh recovers traffic.


## Self-check

- [x] Every section filled — markdown + code
- [x] Notebook runs top to bottom
- [x] No client names, URLs, or private queries
- [x] Claim language: observed / measured / directional / decision-support
- [x] Deployed paper has all 9 sections including **Abstract** and **Acknowledgments** (flyrank.ai)
- [x] `submission/paper_url.txt` holds the exact live URL (one line)
- [x] ML-12: demo outline + social cut + employer summary
- [ ] Committed + submitted repo URL on the card